# BSI reducido en NHANES: ¿suma sobre edad + reloj epigenético + carga alostática? (v3)
NHANES 1999–2002, adultos ≥50 años con metilación. Desenlace: muerte por cualquier causa hasta 2019.

**Cómo usarlo:** *Entorno de ejecución → Ejecutar todo*. Todo se descarga solo del CDC y cada archivo se verifica. Si alguna celda se detiene con un aviso, mándame lo que imprime. Demora unos minutos en total.

**Reglas fijadas ANTES de mirar resultados**
- Reloj: `epi_PhenoAge`.
- Primario: M4 (M3 + BSI promedio) vs M3 (edad + sexo + reloj + carga alostática), ΔAUC fuera de muestra, IC95% por bootstrap pareado. **SUMA** si el límite inferior > 0; **NO SUMA** si el IC cae entero dentro de ±0.01; si no, **INCONCLUSO**.
- Sensibilidad: lo mismo solo con casos completos (los 7 marcadores presentes).
- Control: BSI vs los mismos 7 marcadores como números continuos (M4 vs M6).

Es un BSI **reducido** (inflamación + metabolismo + reloj epigenético; falta cortisol y BDNF). Sirve para valor añadido, no para la capacidad adaptativa (PE3).

In [1]:
%%writefile bsi_nhanes.py
"""Valor anadido del BSI reducido sobre edad + reloj epigenetico + carga alostatica (NHANES 1999-2002, >=50 anos).
Todo lo que sigue esta FIJADO ANTES de mirar resultados (ver DECISION_RULE)."""
import re
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

SEED = 2026
N_BOOT = 2000
MARGIN = 0.01  # margen de equivalencia en AUC
DECISION_RULE = ("SUMA: limite inferior del IC95% de dAUC (M4-M3) > 0. "
                 "NO SUMA: IC95% de dAUC completamente dentro de (-0.01, +0.01). "
                 "Cualquier otro caso: INCONCLUSO.")

# Nombres candidatos de columnas (NHANES). Si alguno falta, check_columns() lo avisa y se detiene.
CANDIDATES = {
    "age": ["RIDAGEYR"], "sex": ["RIAGENDR"],
    "crp": ["LBXCRP", "LBDCRP"], "glucose": ["LBXGLU"], "insulin": ["LBXIN"],
    "tg": ["LBXTR"], "hdl": ["LBDHDL", "LBXHDD"], "hba1c": ["LBXGH"],
    "uma": ["URXUMA"], "ucr": ["URXUCR"],
}
CLOCK_REGEX = r"(pheno|grim|dunedin|hannum|horvath)"

def pick(df, names):
    for n in names:
        if n in df.columns:
            return n
    low = {c.lower(): c for c in df.columns}
    for n in names:
        if n.lower() in low:
            return low[n.lower()]
    return None

def check_columns(df):
    found, missing = {}, []
    for role, names in CANDIDATES.items():
        c = pick(df, names)
        (found.__setitem__(role, c) if c else missing.append(role))
    clocks = [c for c in df.columns if re.search(CLOCK_REGEX, c, re.I)]
    return found, missing, clocks

def zs(x):
    return (x - x.mean()) / x.std(ddof=0)

def build_scores(df, cols, clock_col):
    d = pd.DataFrame({"seqn": df["SEQN"]})
    d["age"] = df[cols["age"]]; d["female"] = (df[cols["sex"]] == 2).astype(int)
    crp, ins = np.log(df[cols["crp"]].clip(lower=1e-3)), np.log(df[cols["insulin"]].clip(lower=0.1))
    tg, glu = np.log(df[cols["tg"]].clip(lower=1)), df[cols["glucose"]]
    hba, hdl = df[cols["hba1c"]], df[cols["hdl"]]
    acr = np.log((df[cols["uma"]].clip(lower=0.1) / df[cols["ucr"]].clip(lower=1)) * 100)
    # reloj: aceleracion = residuo del reloj sobre la edad cronologica
    c = df[clock_col]; ok = c.notna() & d["age"].notna()
    slope, icpt = np.polyfit(d.loc[ok, "age"], c[ok], 1)
    accel = c - (icpt + slope * d["age"])
    # dominios (mayor = peor)
    d["dom_inflam"] = zs(pd.concat([zs(crp), zs(acr)], axis=1).mean(axis=1))
    d["dom_metab"] = zs(pd.concat([zs(glu), zs(ins), zs(tg), zs(hba), -zs(hdl)], axis=1).mean(axis=1))
    d["dom_epi"] = zs(accel)
    dom = d[["dom_inflam", "dom_metab", "dom_epi"]]
    d["bsi_mean"] = dom.mean(axis=1)    # primario: promedio con igual peso
    d["bsi_max"] = dom.max(axis=1)      # secundario: dominio mas alterado
    d["clock_accel"] = accel
    # carga alostatica simplificada: n de marcadores en el cuartil de peor riesgo
    risk = pd.concat([crp > crp.quantile(.75), acr > acr.quantile(.75), hba > hba.quantile(.75),
                      tg > tg.quantile(.75), hdl < hdl.quantile(.25),
                      glu > glu.quantile(.75), ins > ins.quantile(.75)], axis=1)
    d["allo_load"] = risk.sum(axis=1)
    return d

MODELS = {
    "M1 edad+sexo": ["age", "female"],
    "M2 +reloj": ["age", "female", "clock_accel"],
    "M3 +carga alostatica": ["age", "female", "clock_accel", "allo_load"],
    "M4 +BSI (promedio)": ["age", "female", "clock_accel", "allo_load", "bsi_mean"],
    "M5 +BSI (dominio max)": ["age", "female", "clock_accel", "allo_load", "bsi_max"],
}

def oof_predictions(d, y, folds=5, repeats=5):
    """Predicciones fuera de muestra, mismas particiones para todos los modelos."""
    out = {m: np.zeros((repeats, len(d))) for m in MODELS}
    for r in range(repeats):
        for tr, te in StratifiedKFold(folds, shuffle=True, random_state=SEED + r).split(d, y):
            for m, feats in MODELS.items():
                mod = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
                mod.fit(d.iloc[tr][feats], y[tr])
                out[m][r, te] = mod.predict_proba(d.iloc[te][feats])[:, 1]
    return {m: v.mean(axis=0) for m, v in out.items()}

def paired_boot_delta(y, p_new, p_old, n_boot=N_BOOT, seed=SEED):
    rng = np.random.default_rng(seed); n = len(y); ds = []
    for _ in range(n_boot):
        i = rng.integers(0, n, n)
        if y[i].min() == y[i].max():
            continue
        ds.append(roc_auc_score(y[i], p_new[i]) - roc_auc_score(y[i], p_old[i]))
    return np.percentile(ds, [2.5, 97.5])

def run(df_all, cols, clock_col, y_col="MORTSTAT"):
    d = build_scores(df_all, cols, clock_col)
    d["y"] = df_all[y_col].values
    d = d.dropna().reset_index(drop=True)
    y = d["y"].astype(int).values
    P = oof_predictions(d, y)
    rows = [{"modelo": m, "AUC (fuera de muestra)": round(roc_auc_score(y, p), 4)} for m, p in P.items()]
    res = pd.DataFrame(rows)
    out = {"n": len(d), "muertes": int(y.sum()), "tabla": res,
           "corr_dominios": d[["dom_inflam", "dom_metab", "dom_epi"]].corr().round(2),
           "corr_bsi_allo": round(float(d["bsi_mean"].corr(d["allo_load"])), 2)}
    for name, new in [("M4 +BSI (promedio)", "M4 +BSI (promedio)"), ("M5 +BSI (dominio max)", "M5 +BSI (dominio max)")]:
        lo, hi = paired_boot_delta(y, P[new], P["M3 +carga alostatica"])
        dauc = roc_auc_score(y, P[new]) - roc_auc_score(y, P["M3 +carga alostatica"])
        verdict = ("SUMA" if lo > 0 else "NO SUMA" if (lo > -MARGIN and hi < MARGIN) else "INCONCLUSO")
        out[name] = {"dAUC vs M3": round(dauc, 4), "IC95%": (round(lo, 4), round(hi, 4)), "veredicto": verdict}
    return out

Writing bsi_nhanes.py


In [2]:
import os, glob, requests, numpy as np, pandas as pd
import bsi_nhanes as B
os.makedirs('data', exist_ok=True)
for f in glob.glob('data/*'): os.remove(f)

BASE_URL = 'https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/{y}/DataFiles/{n}.xpt'
NAMES = {
 '1999': ['DEMO','LAB11','LAB10AM','LAB13AM','LAB13','LAB10','LAB16'],
 '2001': ['DEMO_B','L11_B','L10AM_B','L13AM_B','L13_B','L10_B','L16_B'],
}
MORT = 'https://ftp.cdc.gov/pub/Health_Statistics/NCHS/datalinkage/linked_mortality/'
MORTFILES = ['NHANES_1999_2000_MORT_2019_PUBLIC.dat','NHANES_2001_2002_MORT_2019_PUBLIC.dat']
DNAM_URL = 'https://wwwn.cdc.gov/nchs/data/nhanes/dnam/dnmepi.sas7bdat'

def es_xport(b): return b[:20].startswith(b'HEADER RECORD')
fallan = []
def bajar(url, dest, valido):
    r = requests.get(url, timeout=120)
    if r.status_code == 200 and valido(r.content):
        open(dest, 'wb').write(r.content); print('OK   ', os.path.basename(dest), len(r.content), 'bytes')
    else:
        fallan.append(os.path.basename(dest)); print('FALLA', os.path.basename(dest), r.status_code, r.content[:40])

for y, ns in NAMES.items():
    for n in ns: bajar(BASE_URL.format(y=y, n=n), f'data/{n}.XPT', es_xport)
for n in MORTFILES:
    bajar(MORT + n, f'data/{n}', lambda b: len(b) > 100000 and not b[:15].lower().startswith(b'<!doc'))
bajar(DNAM_URL, 'data/dnmepi.sas7bdat', lambda b: len(b) > 100000 and not b[:15].lower().startswith(b'<!doc'))
print('\nFallan:', fallan)
assert not fallan, 'Alguna descarga fallo; mandame esta lista.'

OK    DEMO.XPT 11500560 bytes
OK    LAB11.XPT 469040 bytes
OK    LAB10AM.XPT 210960 bytes
OK    LAB13AM.XPT 221040 bytes
OK    LAB13.XPT 335200 bytes
OK    LAB10.XPT 109200 bytes
OK    LAB16.XPT 314240 bytes
OK    DEMO_B.XPT 3273520 bytes
OK    L11_B.XPT 446240 bytes
OK    L10AM_B.XPT 236480 bytes
OK    L13AM_B.XPT 251760 bytes
OK    L13_B.XPT 371920 bytes
OK    L10_B.XPT 120160 bytes
OK    L16_B.XPT 347360 bytes
OK    NHANES_1999_2000_MORT_2019_PUBLIC.dat 487666 bytes
OK    NHANES_2001_2002_MORT_2019_PUBLIC.dat 540362 bytes
OK    dnmepi.sas7bdat 1245184 bytes

Fallan: []


In [3]:
def read_cycle(names):
    base = None
    for n in names:
        t = pd.read_sas(f'data/{n}.XPT', format='xport', encoding='utf-8')
        base = t if base is None else base.merge(t, on='SEQN', how='outer', suffixes=('','_dup'))
    return base
lab = pd.concat([read_cycle(NAMES['1999']), read_cycle(NAMES['2001'])], ignore_index=True)
epi = pd.read_sas('data/dnmepi.sas7bdat', encoding='utf-8')
epi.columns = [c if c == 'SEQN' else 'epi_' + c for c in epi.columns]
cols, miss, clocks = B.check_columns(lab.merge(epi, on='SEQN'))
print('roles que faltan:', miss); print('relojes disponibles:', clocks)
CLOCK = 'epi_PhenoAge'   # fijado antes de ver resultados; no lo cambies
assert not miss and CLOCK in epi.columns, 'Falta una variable o el reloj; mandame lo impreso.'
print('Reloj elegido:', CLOCK)

roles que faltan: []
relojes disponibles: ['epi_HorvathAge', 'epi_HannumAge', 'epi_PhenoAge', 'epi_GrimAgeMort', 'epi_GrimAge2Mort', 'epi_HorvathTelo', 'epi_DunedinPoAm']
Reloj elegido: epi_PhenoAge


In [4]:
spec = [(0,14),(14,15),(15,16),(16,19),(43,46),(46,49)]
names = ['SEQN','ELIGSTAT','MORTSTAT','UCOD','PERMTH_INT','PERMTH_EXM']
m = pd.concat([pd.read_fwf(f'data/{f}', colspecs=spec, names=names) for f in MORTFILES], ignore_index=True)
for c in ['SEQN','ELIGSTAT','MORTSTAT']:
    m[c] = pd.to_numeric(m[c], errors='coerce')
m = m[m['ELIGSTAT'] == 1]
print('Mortalidad:', m['MORTSTAT'].value_counts(dropna=False).to_dict())
assert set(m['MORTSTAT'].dropna().unique()) <= {0,1}, 'El ancho fijo no calza; mandame esto.'
df = lab.merge(epi, on='SEQN').merge(m[['SEQN','MORTSTAT']], on='SEQN')
df = df[df[cols['age']] >= 50]
print('Participantes >=50 con metilacion y mortalidad:', len(df), '| muertes:', int(df['MORTSTAT'].sum()))

Mortalidad: {0.0: 8133, 1.0: 3299}
Participantes >=50 con metilacion y mortalidad: 4446 | muertes: 2493


## Análisis primario (regla fijada de antemano)

In [5]:
res = B.run(df, cols, CLOCK)
print(f"n analizado = {res['n']}, muertes = {res['muertes']}"); display(res['tabla'])
print('Correlacion entre dominios:'); display(res['corr_dominios'])
print('Correlacion BSI vs carga alostatica:', res['corr_bsi_allo'])
for k in ('M4 +BSI (promedio)','M5 +BSI (dominio max)'): print(k, res[k])
print('\nREGLA:', B.DECISION_RULE)
res['tabla'].to_csv('tabla_auc.csv', index=False)

n analizado = 2532, muertes = 1361


,modelo,AUC (fuera de muestra)
0,M1 edad+sexo,0.8127
1,M2 +reloj,0.8216
2,M3 +carga alostatica,0.8277
3,M4 +BSI (promedio),0.8317
4,M5 +BSI (dominio max),0.8352


Correlacion entre dominios:


,dom_inflam,dom_metab,dom_epi
dom_inflam,1.00,0.29,0.18
dom_metab,0.29,1.00,0.08
dom_epi,0.18,0.08,1.00


Correlacion BSI vs carga alostatica: 0.65
M4 +BSI (promedio) {'dAUC vs M3': np.float64(0.0039), 'IC95%': (np.float64(0.0012), np.float64(0.0069)), 'veredicto': 'SUMA'}
M5 +BSI (dominio max) {'dAUC vs M3': np.float64(0.0074), 'IC95%': (np.float64(0.0034), np.float64(0.0116)), 'veredicto': 'SUMA'}

REGLA: SUMA: limite inferior del IC95% de dAUC (M4-M3) > 0. NO SUMA: IC95% de dAUC completamente dentro de (-0.01, +0.01). Cualquier otro caso: INCONCLUSO.


## Control y sensibilidad (solo casos completos: los 7 marcadores presentes)
Nota: en el análisis primario, si faltaba un marcador, el dominio se promedió con los disponibles y la carga alostática lo contó como 'no riesgoso'. Aquí se exige que estén los 7, por eso `n` es menor.

In [6]:
#   "No agrega" si el IC95% cae entero dentro de (-0.01, +0.01). Otro caso: INCONCLUSO.
import numpy as np, pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

d = B.build_scores(df, cols, CLOCK)
d["y"] = df["MORTSTAT"].values
d["l_crp"] = np.log(df[cols["crp"]].clip(lower=1e-3)).values
d["l_acr"] = np.log((df[cols["uma"]].clip(lower=0.1) / df[cols["ucr"]].clip(lower=1)) * 100).values
d["hba1c"] = df[cols["hba1c"]].values
d["l_tg"] = np.log(df[cols["tg"]].clip(lower=1)).values
d["hdl"] = df[cols["hdl"]].values
d["glu"] = df[cols["glucose"]].values
d["l_ins"] = np.log(df[cols["insulin"]].clip(lower=0.1)).values
d = d.dropna().reset_index(drop=True)
y = d["y"].astype(int).values
print("n (casos completos) =", len(d), "| muertes =", int(y.sum()), "(menor que el primario porque se exigen los 7 marcadores)")

BASE = ["age", "female", "clock_accel"]
MK = ["l_crp", "l_acr", "hba1c", "l_tg", "hdl", "glu", "l_ins"]
MODELS2 = {
    "M3 +carga alostatica":              BASE + ["allo_load"],
    "M4 +BSI (promedio)":                BASE + ["allo_load", "bsi_mean"],
    "M6 +marcadores continuos":          BASE + ["allo_load"] + MK,
    "M7 +3 dominios por separado":       BASE + ["allo_load", "dom_inflam", "dom_metab", "dom_epi"],
}
def oof(d, y, folds=5, repeats=5):
    out = {m: np.zeros((repeats, len(d))) for m in MODELS2}
    for r in range(repeats):
        for tr, te in StratifiedKFold(folds, shuffle=True, random_state=B.SEED + r).split(d, y):
            for m, f in MODELS2.items():
                mod = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))
                mod.fit(d.iloc[tr][f], y[tr]); out[m][r, te] = mod.predict_proba(d.iloc[te][f])[:, 1]
    return {m: v.mean(axis=0) for m, v in out.items()}
P = oof(d, y)
print(pd.DataFrame([{"modelo": m, "AUC (fuera de muestra)": round(roc_auc_score(y, p), 4)} for m, p in P.items()]).to_string(index=False))

def veredicto(lo, hi):
    return "AGREGA" if lo > 0 else "NO AGREGA" if (lo > -B.MARGIN and hi < B.MARGIN) else "INCONCLUSO"
for nuevo, viejo, titulo in [("M4 +BSI (promedio)", "M6 +marcadores continuos", "PRIMARIO  BSI sobre marcadores continuos (M4 - M6)"),
                             ("M6 +marcadores continuos", "M3 +carga alostatica", "M6 sobre carga alostatica (M6 - M3)"),
                             ("M7 +3 dominios por separado", "M4 +BSI (promedio)", "3 dominios por separado vs BSI promedio (M7 - M4)")]:
    lo, hi = B.paired_boot_delta(y, P[nuevo], P[viejo])
    dauc = roc_auc_score(y, P[nuevo]) - roc_auc_score(y, P[viejo])
    print(f"{titulo}: dAUC = {dauc:.4f}  IC95% = ({lo:.4f}, {hi:.4f})  -> {veredicto(lo, hi)}")


# Sensibilidad: el mismo analisis primario (M4 vs M3) en casos completos
lo, hi = B.paired_boot_delta(y, P["M4 +BSI (promedio)"], P["M3 +carga alostatica"])
dauc = roc_auc_score(y, P["M4 +BSI (promedio)"]) - roc_auc_score(y, P["M3 +carga alostatica"])
print(f"M4 - M3 (solo casos completos, n={len(y)}): dAUC = {dauc:.4f}  IC95% = ({lo:.4f}, {hi:.4f})  -> {veredicto(lo, hi)}")

n (casos completos) = 1228 | muertes = 644 (menor que el primario porque se exigen los 7 marcadores)
                     modelo  AUC (fuera de muestra)
       M3 +carga alostatica                  0.8281
         M4 +BSI (promedio)                  0.8316
   M6 +marcadores continuos                  0.8373
M7 +3 dominios por separado                  0.8368
PRIMARIO  BSI sobre marcadores continuos (M4 - M6): dAUC = -0.0057  IC95% = (-0.0128, 0.0011)  -> INCONCLUSO
M6 sobre carga alostatica (M6 - M3): dAUC = 0.0092  IC95% = (0.0015, 0.0177)  -> AGREGA
3 dominios por separado vs BSI promedio (M7 - M4): dAUC = 0.0052  IC95% = (0.0001, 0.0108)  -> AGREGA
M4 - M3 (solo casos completos, n=1228): dAUC = 0.0036  IC95% = (-0.0004, 0.0079)  -> NO AGREGA


## Cómo leer esto
- **Primario**: ¿el BSI promedio mejora la predicción sobre edad, reloj y carga alostática? Mirar el veredicto y el tamaño de ΔAUC (no solo el veredicto).
- **Control (M4 − M6)**: si el BSI no supera a los mismos marcadores continuos, la mejora se explica por usar los marcadores como números.
- **Límites**: desenlace binario sin tiempo hasta el evento, sin pesos muestrales, IC bootstrap que no incluye la variabilidad del ajuste, PhenoAge comparte marcadores con el BSI, y solo mide valor añadido en una foto transversal: **no prueba capacidad adaptativa (PE3)**.